# Solar filament instance segmentation: full pipeline

Filament Segmentation Challenge 2026 (IEEE BigData Cup). This notebook reproduces the final
submission from the weights stored in this repository, without retraining:

1. **Preprocessing:** solar-disk mask, limb-darkening flattening, robust normalisation
2. **Model:** 5 U-Net / ResNet34 models (one per date-grouped fold), probabilities averaged
3. **Inference:** overlapping 1024 px tiles with Hann blending and flip TTA, full 2048 px resolution
4. **Post-processing:** threshold, fragment grouping by morphological closing, size and confidence filtering
5. **Submission:** disjoint masks encoded as COCO RLE, validated before writing

Training code lives in `src/filament/train.py` and `scripts/train.py` (see README).

## 0. Setup
Set `DATA_ROOT` to the unzipped Kaggle data folder (the one containing `test/test_images`).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from filament.preprocess import preprocess_file, read_gray
from filament.model import load_checkpoint
from filament.infer import predict_image
from filament.postprocess import PostCfg, instances
from filament.submission import to_dataframe, validate, write
from filament.rle import decode

DATA_ROOT = Path("~/data/MAGFiLO_1.0_Kaggle_2026").expanduser()   # <-- change if needed
TEST_DIR = DATA_ROOT / "test" / "test_images"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO, "| device:", DEVICE, "| test images:", len(list(TEST_DIR.glob('*.jp*g'))))

## 1. Final configuration

The five checkpoints are the best epoch (by validation PQ) of each cross-validation fold.
Post-processing parameters were tuned once on the out-of-fold predictions of all 707 training
images (`scripts/tune_postprocess.py`), scored with the official Panoptic Quality.

In [ ]:
CHECKPOINTS = [REPO / "weights" / f for f in
               ["r34_f0_e80.pt", "r34_f1.pt", "r34_f2.pt", "r34_f3.pt", "r34_f4.pt"]]
with open(REPO / "configs" / "final_post.json") as f:
    POST = PostCfg(**json.load(f)["post"])

models, cfgs = zip(*[load_checkpoint(p, DEVICE) for p in CHECKPOINTS])
CFG = cfgs[0]
ICFG = CFG["infer"]
print("preprocess:", CFG["preprocess"], "| model:", CFG["model"]["arch"], CFG["model"]["encoder"])
print("inference:", ICFG)
print("post-processing:", POST)

## 2. Preprocessing
Left: raw GONG H-alpha image. Right: model input (disk masked, limb darkening removed).

In [ ]:
example = sorted(TEST_DIR.glob("*.jp*g"))[0]
raw = read_gray(example)
pre = preprocess_file(example, CFG["preprocess"])
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(raw, cmap="gray"); ax[0].set_title(f"raw: {example.name}")
ax[1].imshow(pre, cmap="gray"); ax[1].set_title("preprocessed")
for a in ax: a.axis("off")
plt.show()

## 3. Inference and post-processing on the example image

In [ ]:
prob = predict_image(list(models), pre, tile=ICFG["tile"], overlap=ICFG["overlap"],
                     tta=ICFG.get("tta", True), batch=ICFG["batch"], device=DEVICE)
inst = instances(prob, POST)
overlay = np.zeros(prob.shape, np.int32)
for k, c in enumerate(inst, 1):
    overlay[decode(c["counts"]).astype(bool)] = k

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(pre, cmap="gray"); ax[0].set_title("input")
ax[1].imshow(prob, cmap="magma", vmin=0, vmax=1); ax[1].set_title("ensemble probability")
ax[2].imshow(pre, cmap="gray"); ax[2].imshow(np.ma.masked_equal(overlay, 0), cmap="tab20", alpha=0.7)
ax[2].set_title(f"{len(inst)} filament instances")
for a in ax: a.axis("off")
plt.show()

## 4. Full test set and submission file

In [ ]:
files = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
preds, t0 = {}, time.time()
for i, p in enumerate(files):
    img = preprocess_file(p, CFG["preprocess"])
    pr = predict_image(list(models), img, tile=ICFG["tile"], overlap=ICFG["overlap"],
                       tta=ICFG.get("tta", True), batch=ICFG["batch"], device=DEVICE)
    preds[p.stem] = [c["counts"] for c in instances(pr, POST)]
    if (i + 1) % 30 == 0:
        print(f"{i + 1}/{len(files)} images, {time.time() - t0:.0f} s")
dt = time.time() - t0

sub = to_dataframe(preds)
problems = validate(sub)
assert not problems, problems
write(sub, REPO / "submission.csv")
print(f"{len(sub)} filaments on {sum(bool(v) for v in preds.values())}/{len(files)} images")
print(f"{dt:.0f} s total, {dt / len(files):.2f} s/image on {DEVICE}")
sub.head()

## 5. Results

| Model | Out-of-fold PQ (707 images) | Public LB |
| --- | --- | --- |
| 5 x U-Net ResNet34 (this notebook) | 0.448 (95% CI 0.440-0.457, fold std 0.009) | 0.39 |

On the 296 training images with 2 to 3 annotators, one annotator scored against another reaches
PQ 0.340, while the model scored against each annotator reaches 0.447 (`scripts/eda_oof.py`).
See the technical report for the experiments and error analysis.